# SmolLM2-135M → AM-CeNN + 8-Shard Top-2 FFN

This notebook tests the TinyCeNN hypothesis on a real 30-layer SmolLM2 decoder.

**Student architecture**
- pretrained SmolLM2 embeddings, RMSNorms and tied LM head stay intact;
- every Transformer self-attention block is replaced by recurrent associative memory:
  `S_t = S_{t-1} + φ(k_t)v_tᵀ`, `z_t = z_{t-1}+φ(k_t)`,
  `a_t = φ(q_t)ᵀS_t / (φ(q_t)ᵀz_t+ε)`;
- `φ` uses finite positive random features approximating the softmax kernel;
- every 1536-wide SwiGLU FFN is split into **8 × 192** disjoint shards with **Top-2** routed correction;
- the complete FFN is preserved exactly at `route_mix=0`;
- no `T×T` self-attention matrix is built in the student.

Training is a fast teacher-distillation run and deliberately skips long evaluation.


In [ ]:
import subprocess, sys, pathlib, importlib, json, torch
subprocess.run(["nvidia-smi"], check=False)

REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "huggingface_hub"], check=True)
SRC = REPO_DIR / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
import tinycenn_lm
print("TinyCeNN import:", tinycenn_lm.__file__)


## Hugging Face login

Add a Hugging Face **write token** to Colab Secrets as `HF_TOKEN`.


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add HF_TOKEN to Colab Secrets first.")
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()["name"]
print("HF user:", HF_USER)


## Settings


In [ ]:
BASE_MODEL = "HuggingFaceTB/SmolLM2-135M"
TARGET_REPO = f"{HF_USER}/SmolLM2-135M-AMCeNN-Top2"
OUTPUT_DIR = REPO_DIR / "checkpoints" / "smollm2-amcenn-top2"

MAX_TOKENS = 2_000_000
MAX_RUNTIME_MINUTES = 45
CONTEXT_LENGTH = 128
FEATURE_DIM = 32

print("base:", BASE_MODEL)
print("target:", TARGET_REPO)


## Train the attention-free student

Only the AM-CeNN Q/K/V/O projections and tiny Top-2 routers/mix scalars train by default.
The exact pretrained FFN shards remain frozen. No held-out benchmark is run.


In [ ]:
cmd = [
    sys.executable, str(REPO_DIR / "scripts" / "train_smollm2_amcenn.py"),
    "--base-model", BASE_MODEL,
    "--output-dir", str(OUTPUT_DIR),
    "--context-length", str(CONTEXT_LENGTH),
    "--feature-dim", str(FEATURE_DIM),
    "--max-tokens", str(MAX_TOKENS),
    "--max-runtime-minutes", str(MAX_RUNTIME_MINUTES),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## Inspect the training report


In [ ]:
report = json.loads((OUTPUT_DIR / "smollm2_amcenn_training_report.json").read_text())
print(json.dumps(report, indent=2))


## Publish the new model


In [ ]:
api.create_repo(TARGET_REPO, repo_type="model", exist_ok=True)
api.upload_folder(
    repo_id=TARGET_REPO,
    repo_type="model",
    folder_path=str(OUTPUT_DIR),
    commit_message="Publish SmolLM2 AM-CeNN Top-2 student",
)
print("Published:", f"https://huggingface.co/{TARGET_REPO}")


## Reload from Hugging Face and test immediately

This is intentionally only a structural + generation smoke test, not a long evaluation.


In [ ]:
from transformers import AutoTokenizer
from tinycenn_lm.smollm2_amcenn import (
    AMCeNNAttention,
    ShardedTop2LlamaMLP,
    build_smollm2_amcenn,
)

REMOTE_DIR = pathlib.Path(snapshot_download(TARGET_REPO, token=HF_TOKEN))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = (
    torch.bfloat16
    if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type == "cuda" else torch.float32)
)
model = build_smollm2_amcenn(REMOTE_DIR, device=device, dtype=dtype)
tokenizer = AutoTokenizer.from_pretrained(REMOTE_DIR, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

am_layers = sum(isinstance(m, AMCeNNAttention) for m in model.modules())
moe_layers = sum(isinstance(m, ShardedTop2LlamaMLP) for m in model.modules())
transformer_attention = [m.__class__.__name__ for m in model.modules() if "LlamaAttention" in m.__class__.__name__]

print("AM-CeNN layers:", am_layers)
print("8-shard Top-2 FFNs:", moe_layers)
print("Transformer self-attention remaining:", transformer_attention)
assert am_layers == 30
assert moe_layers == 30
assert not transformer_attention
print("STRUCTURE: PASS")


In [ ]:
prompts = [
    "The capital of Austria is",
    "Artificial intelligence can help",
    "Once upon a time, a small robot was lost in a park.",
    "A small language model can",
]

model.eval()
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            min_new_tokens=20,
            do_sample=True,
            temperature=0.80,
            top_p=0.90,
            top_k=50,
            repetition_penalty=1.10,
            no_repeat_ngram_size=4,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    print("\n" + "=" * 90)
    print(generated)


## What to send back

Send the printed `smollm2_amcenn_training_report.json` and the four generations.
The first question is whether a **30-layer attention-free recurrent associative-memory student** can recover useful language behavior while the FFN conversion remains function-preserving.
